In [ ]:
from datasets import load_dataset_builder, load_dataset
from huggingface_hub import login
import polars as pl
import io
import librosa
from pydantic import BaseModel
from theogradz.models.models import AudioData, LibriSpeechSample
from IPython.display import Audio

login()

In [ ]:
builder = load_dataset_builder("openslr/librispeech_asr")

In [ ]:
# gb
int(builder.info.dataset_size or 0) / 1_000_000_000

In [ ]:
ds_stream = load_dataset(
    "openslr/librispeech_asr", split="train.clean.100", streaming=True
)

In [ ]:
subset = ds_stream.take(100)

In [ ]:
subset.features

In [ ]:
df: pl.DataFrame = pl.from_pandas(subset.to_pandas())  # type: ignore

In [ ]:
df

In [ ]:
raw_row = df.row(0, named=True)

In [ ]:
for k, v in raw_row.items():
    print(f"key: {k}, value type: {type(v)}")

In [ ]:
sample = LibriSpeechSample.model_validate(raw_row)

In [ ]:
data, sr = sample.audio.load()

In [ ]:
data.dtype, data.shape

In [ ]:
Audio(data, rate=sr)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


class CausalConv1d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, stride: int = 1):
        super().__init__()
        self.pad = kernel_size - 1
        self.conv = nn.Conv1d(
            in_ch, out_ch, kernel_size, padding=0
        )  # left padding handled in forward pass

    def forward(self, x: Tensor):
        x = F.pad(x, (self.pad, 0))  # (L, R) padding
        print(f"new x: {x}")
        return self.conv(x)


In [25]:
test_tens = Tensor([[1, 2, 3, 4]])


In [26]:
conv = CausalConv1d(1, 1, 3)

In [27]:
conv.forward(test_tens)

new x: tensor([[0., 0., 1., 2., 3., 4., 0., 0.]])


tensor([[-0.8217, -1.3945, -1.6982, -2.0018, -1.0611,  0.5041]],
       grad_fn=<SqueezeBackward1>)